# 0.5 Physical gratings and the Helmholtz boundary-value problem

An absorbing grating has clear openings between material ridges. We know its shape and material, but we do not yet know the optical field around it. What field is reflected, and what field emerges below the grating?

In [0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb), a thin mask prescribed the field after an aperture. In [0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb), we followed a known field through space. Here we specify the material and the incoming wave, then calculate the field they produce together.

## Motivation and prerequisites

Recall amplitude and phase in $U$, the Helmholtz equation from [0.1](0_1_Scalar_Fields_and_Helmholtz.ipynb), and superposition and outgoing conditions from [0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb). We also use the wave components introduced in [0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb). The lesson explains how these ideas enter a scattering calculation; no experience with a boundary-value solver is needed.

Begin with the material map and compare it with the calculated field. Then follow a few wave components through the layer. These pictures show what the material, illumination and boundary conditions each contribute. The Fourier-method derivation comes afterwards and is optional. Chapter 0 introduces the physical ideas; later chapters develop their mathematics in more detail.

In [0.6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb), we will describe the calculated field using individual diffraction orders and follow their propagation.

:::{note} Model and wavelength conventions

The input is the **vacuum** wavelength $\lambda_0$, named `wavelength` in the code. Air fills the regions above and below the grating, with $n_i=n_t=1$. Inside the layer, the clear openings have index $1$ and the absorber has index $1.5+0.8i$. These describe the layer's materials, not a different index for the surrounding air. The solver uses $k_0=2\pi/\lambda_0$ and the permittivity at each position. See the [course medium convention](0_0_Foundations_of_Scalar_Optics.ipynb#medium-and-wavelength).

The model is deterministic and scalar, and the grating is uniform along $y$. We use time dependence $e^{-i\omega t}$ and transmission toward **negative $z$**. All lengths are in micrometres; $\lambda_0$ is the vacuum wavelength. This reverses the drawing direction used in [0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb), while keeping the same time convention.
:::

Run the cells in order within this notebook; no earlier notebook needs to be running.


:::{note} Interactive examples
Click the **power button** near the top of the page to activate the examples. Binder supplies Python in your browser; no local installation is needed. Wait for the session to connect and the scattering calculation to finish. Use **Material** to switch between real index and extinction, **Show** to inspect field quantities, and **Orders m** to select harmonics. These controls inspect the same solved grating. Use **Reset this view** to restore each starting view. The text, equations and saved figures can be read without activating a session.
:::

For local use, download the {download}`numerical explorer <downloads/grating_physics_numerical_explorer.zip>` or the complete {download}`worked-example scripts <downloads/grating_physics_scripts.zip>` with their shared helpers.

In [ ]:
from pathlib import Path
import sys
import importlib
from IPython.display import display

_support_dir = next((base for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))

# Release current explorers and any remaining objects from earlier notebook versions.
for _name in ('absorber_comparison', 'binary_masks', 'grating_comparison'):
    if _name in globals():
        globals()[_name].close()
if 'grating_views' in globals():
    for previous in grating_views.values():
        previous.close()

# Refresh dependencies before their consumers, including a cached pre-absorber solver.
# Import and reload one module at a time: importing all first can fail on new symbols.
for _module_name in ('scalar_grating', 'binary_transmission', 'thin_grating',
                     'absorbing_grating', 'scalar_grating_view', 'amplitude_grating',
                     'grating_comparison_view', 'absorbing_grating_view'):
    importlib.reload(importlib.import_module(f'wave_optics.{_module_name}'))

import wave_optics.absorbing_grating as absorbing_model
import wave_optics.scalar_grating_view as grating_view
import wave_optics.amplitude_grating as amplitude_model
import wave_optics.absorbing_grating_view as absorbing_view

# One shared physical absorber: centred air opening, air half-spaces.
# Compute 49 harmonics; display only orders -7,...,+7 in coefficient plots.
grating_config = absorbing_model.AbsorbingGratingConfiguration(M=24)
display_order_limit = 7
grating_solution = absorbing_model.solve_absorbing_grating(grating_config)
grating_data = grating_view.ScalarGratingData(grating_solution)
grating_views = {}

def show_grating(stage):
    if stage in grating_views:
        grating_views[stage].close()
    grating_views[stage] = grating_view.ScalarGratingExplorer(
        stage, grating_data, display_order_limit=display_order_limit)
    display(grating_views[stage].ui)


## 1. What physical inputs select the field?

A plane wave arrives normally from above. It illuminates a periodic **binary amplitude transmission grating**: absorbing ridges alternate with clear air openings. The ridges occupy $-h<z<0$. Air also fills the regions above and below the layer, and there is no substrate.

Here **binary** refers to the two material regions. An ideal mask would assign them transmission levels 0 and 1. We instead calculate the field in and around the physical structure, so its values need not be exactly 0 or 1. **Transmission** identifies the output side of the experiment.

The centred clear opening has width $W_{\mathrm{open}}=0.75\,\mu\mathrm m$ and period $d=1.50\,\mu\mathrm m$, so the open fraction is $D_{\mathrm{open}}=W_{\mathrm{open}}/d=0.5$. Initially $h=0.30\,\mu\mathrm m$, $\lambda_0=0.633\,\mu\mathrm m$ and the absorber index is $n_a+i\kappa_a=1.5+0.8i$. These are illustrative optical-scale parameters, not fitted chrome data.

**Predict:** does specifying the material also specify the illumination?

**Explore:** locate the air opening, absorbing ridges and output region. Switch the material map between real index $n$ and extinction coefficient $\kappa$.

The layer has finite thickness, but the grating repeats indefinitely along x and is uniform along y. The incident plane wave is also extended. The drawing shows only a patch of this repeating structure; it does not represent a finitely illuminated grating.
- Lateral ends, a laser-beam envelope, roughness and fabrication variations are omitted; the stated material index and ideal interfaces are model inputs. [Maystre, §§2.2–2.3](#ref-0-5-maystre) [\[1\]](#ref-0-5-maystre). 

In [ ]:
show_grating('geometry')

The complex material index $\widetilde n(x,z)=n(x,z)+i\kappa(x,z)$ tells us what material is present at each position. We must also specify the incoming wave:

$$U_i=e^{-ik_0z},\qquad k_0=\frac{2\pi}{\lambda_0}.$$

At $z=0$, this wave has amplitude one and phase zero. The **total complex field**, including the scattered field, satisfies

$$\boxed{[\partial_x^2+\partial_z^2+k_0^2\varepsilon(x,z)]U=0,\qquad
\varepsilon=(n+i\kappa)^2.}$$

For our time convention, $\kappa\geq0$ describes a passive absorber. A forward wave in uniform material accumulates $e^{ik_0(n+i\kappa)h}$ over distance $h$: it changes phase and decays in amplitude.

At each horizontal interface, the scalar model matches $U$ and $\partial_zU$. Reflected and transmitted waves must leave the grating, and evanescent contributions must decay away from it. There is no additional incoming wave. Together, these conditions define the **Helmholtz boundary-value problem (HBVP)**. They select the field; we have not assigned a transmission function to the physical grating.

### Read the boundary conditions physically

The material map says where the wave equation changes. It does not, by itself, say which wave arrives or how strongly it illuminates the grating. Here the incoming plane wave fixes that missing input. The field must then satisfy the same scattering problem in all three regions: incident air, structured layer and transmitted air.

At an interface, the scalar matching conditions connect the fields on its two sides. Reflection is an allowed part of the answer, so the field immediately above the layer need not equal the incident wave alone. Below the layer we admit waves leaving the grating and near-field contributions decaying away from it. We do not add illumination arriving from below.

This applies the idea from [0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb): the physical conditions select a solution. A thin-mask model supplies the output field directly. Here we find it by solving the wave equation with the material and boundary conditions. An absorbing ridge can transmit some field and reflect some field while dissipating energy. Calling it an amplitude grating does not require its actual transmitted field to be exactly zero or one.

**Check the distinction:** the material index is an input; the spatial amplitude and phase of $U$ are outputs. Changing the plotted quantity shows another aspect of the same calculated field. It does not change the illumination or material.


## 2. What field does the solver calculate?

**Predict:** does an absorbing ridge simply set the transmitted field to zero, or can reflection and scattering change the field around the opening too?

**Explore:** compare the material map with the solved field. Switch among real field, amplitude, intensity and phase; also inspect the region above the layer.


In [ ]:
show_grating('field')

Above the grating, $U=U_i+U_r$: incident and reflected contributions can form a varying pattern. Inside the layer, the field responds to the periodic material and loses energy in the absorber. Both amplitude and phase can change across the output, even though the intended function is amplitude modulation. Below it, only outgoing transmitted contributions remain.

The views answer different questions:

| Quantity | What it shows |
| --- | --- |
| $\operatorname{Re}U$ | Signed field at $t=0$ under our time convention. |
| $\lvert U\rvert$ | Field amplitude. |
| $\lvert U\rvert^2$ | Relative scalar intensity; this is not an order efficiency. |
| $\arg U$ | Wrapped phase, from $-\pi$ to $\pi$. |

All four views come from the field selected by the boundary conditions. We have **not prescribed a thin-mask transmission function**.

The calculation retains $m=-24,\ldots,+24$ (49 harmonics). This gives a **truncated numerical solution**: the local field must be checked for convergence by increasing the number of harmonics. The material picture shows the prescribed geometry, while the field picture shows this numerical approximation.

:::{dropdown} Numerical and display settings

Coefficient plots and order selectors show only $m=-7,\ldots,+7$ for clarity. The reconstructed fields and power totals still use all 49 harmonics. In the setup cell, `display_order_limit` controls the displayed range independently of the solver cutoff `M`.

Pixels use nearest-neighbour display. Phase is omitted wherever the amplitude is below $10^{-6}$ of the incident amplitude.
:::

Changing the view changes what we inspect, not the solution.

At the outgoing side of $z_e=-h$, write this solution as $U_{\mathrm{exit}}(x)=U(x,z_e)$. Its coefficients are $C_m(z_e)=F_m(z_e)$. This is the same exit-field notation used for a prescribed mask; the physical model determines its values.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mp}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


### Checkpoint: material value or field value?

**Predict:** must the field in an air opening equal 1, and the field in an absorbing ridge equal 0?

**Try it in code:** change `try_x_fraction` between `0.0` and `0.4`. The point lies halfway through the layer. Then set `try_depth_fraction = 1.2` to sample below it. This reads the existing solution.


In [ ]:
import numpy as np

try_x_fraction = 0.0  # x/d: 0 is the opening centre; 0.4 lies in a ridge.
try_depth_fraction = 0.5  # Downward depth divided by h.
try_x = try_x_fraction * grating_config.period
try_z = -try_depth_fraction * grating_config.thickness
try_index = grating_config.refractive_index([try_x], [try_z])[0, 0]
try_u = grating_solution.reconstruct([try_x], [try_z])[0, 0]
try_phase = f"{np.angle(try_u):+.3f}" if abs(try_u) >= 1e-6 else "undefined"
print(f"Point (x, z) = ({try_x:+.3f}, {try_z:+.3f}) um")
print(f"Material index = {try_index:.3f}; complex field U = {try_u:.3f}")
print(f"|U| = {abs(try_u):.3f}; arg U = {try_phase} rad")


:::{dropdown} Check your prediction


The index identifies the material at the point; it is not the field. The calculated field includes the effects of illumination and scattering throughout the structure. Even in an air opening its amplitude need not be 1. A phase is meaningful only where the field amplitude is appreciable.


:::


## 3. How does the same problem look in Fourier space?

Under normal illumination, the field repeats with period $d$. At each depth, we describe its variation across the grating by adding Fourier harmonics:

$$U(x,z)=\sum_{m=-M}^{M}F_m(z)e^{imGx},\qquad G=\frac{2\pi}{d}.$$

Each $F_m(z)$ is an **unknown complex function**: it describes how one harmonic changes in amplitude and phase with depth. The scattering calculation determines these functions.

**Predict:** can each harmonic evolve independently while it passes through the periodic material?

**Explore:** compare $m=0$, $1$ and $3$ through the shaded layer. Select other indices with Ctrl/Cmd-click. The graph shows magnitudes; the complex functions also carry phase.

:::{dropdown} Notation used by the solver

We write the complex functions as $F_m(z)$ and reserve $A_m$ for nonnegative amplitude magnitudes. The vector $\mathbf A$ and backend methods retain their existing matrix/API names.
:::


At a chosen depth, $F_m(z)$ gives the complex amplitude of harmonic $m$. In the uniform air below the grating, [0.6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb) names its value at a selected plane $C_m(z_{\rm ref})=F_m(z_{\rm ref})$. Inside the patterned layer, the functions $F_m$ affect one another. They cannot be propagated as independent order waves.

In [ ]:
show_grating('harmonics')

In either air half-space, each harmonic obeys its own equation,

$$F_m''+(k_0^2-k_{x,m}^2)F_m=0,\qquad k_{x,m}=mG.$$

Inside the layer the complex permittivity is periodic, $\varepsilon(x)=\sum_q\varepsilon_q e^{iqGx}$. Multiplying $U$ by this material profile mixes the harmonics, giving

$$\boxed{\mathbf A''(z)+\mathbf Q\mathbf A(z)=0.}$$

**Distinguish three descriptions:**

- **Fourier harmonics:** $F_m(z)$ are coupled inside the patterned material.
- **Layer eigenmodes:** certain combinations of harmonics evolve independently within a layer whose material profile is uniform with depth. Diagonalizing the coupling matrix finds these combinations. Matching the fields at the interfaces sets how much of each mode is present.
- **Exterior orders:** in a homogeneous region, the Fourier harmonics decouple and propagate or decay independently.

This is the **Scalar Fourier Modal Method (FMM)**: expand the material and field in Fourier harmonics, solve the coupled modes within the layer, and match them at the interfaces. The independently evolving modes inside the layer are combinations of harmonics, rather than individual harmonics.

FMM is also widely called **rigorous coupled-wave analysis (RCWA)** in electromagnetic grating calculations ([Lalanne and Hugonin, 2006](#ref-0-5-lalanne-hugonin) [\[2\]](#ref-0-5-lalanne-hugonin)). Here we apply that Fourier-modal framework to scalar Helmholtz with scalar interface matching. A full vector RCWA formulation additionally resolves Maxwell fields and polarization; it is not implemented here.

:::{dropdown} Optional derivation: how does the absorbing geometry enter Q?

Substitution into Helmholtz gives

$$F_m''-k_{x,m}^2F_m+k_0^2\sum_p\varepsilon_{m-p}F_p=0,\qquad
Q_{mp}=k_0^2\varepsilon_{m-p}-k_{x,m}^2\delta_{mp}.$$

Let $R_{\mathrm{open}}(x)=1$ in the centred air opening and zero in the absorber. Then

$$\varepsilon(x)=\varepsilon_a+(1-\varepsilon_a)R_{\mathrm{open}}(x),\qquad
\varepsilon_a=(n_a+i\kappa_a)^2,$$

$$\varepsilon_q=\varepsilon_a\delta_{q0}+(1-\varepsilon_a)D_{\mathrm{open}}\,\operatorname{sinc}(qD_{\mathrm{open}}),\qquad
\operatorname{sinc}(v)=\frac{\sin(\pi v)}{\pi v}.$$

The air exterior has only $\varepsilon_0=1$, making $\mathbf Q$ diagonal there. The absorbing layer has off-diagonal coupling and a generally non-Hermitian complex matrix. The solver uses a general eigensystem and a bounded modal basis. The interface conditions select the complex weights.

:::

Writing the field as Fourier harmonics changes how we calculate it. The spatial wave equation and the coupled harmonic equations still describe the same absorbing grating.

For the complete derivation of the implemented solver, including its boundary-matching matrix and numerical checks, see [Appendix_1](Appendix_1_Fourier_Modal_Method_for_Scalar_Grating_Scattering.ipynb).


### Checkpoint: does one incident wave mean one harmonic?

**Predict:** can the solved field contain $m=1$ when the normally incident wave has $m=0$?

**Try it in code:** choose `try_m = 0`, `1` or `-1`. Read the harmonic above, inside and below the layer. Changing the selected index inspects the same solution; it does not change the illumination.


In [ ]:
import numpy as np

try_m = 1
try_j = int(np.flatnonzero(grating_solution.orders == try_m)[0])
try_h = grating_config.thickness
try_z_samples = np.array([0.10, -try_h / 2, -try_h - 0.10])
try_amplitudes = grating_solution.harmonics(try_z_samples)[:, try_j]
print("z (um)       Re F_m       Im F_m        |F_m|")
for try_z, try_a in zip(try_z_samples, try_amplitudes):
    print(f"{try_z:+.3f}      {try_a.real:+.5f}     {try_a.imag:+.5f}     {abs(try_a):.5f}")


:::{dropdown} Check your prediction


The periodic material couples the incident harmonic to others. Above the layer, a nonzero $m=1$ contribution belongs to the reflected field; below it, to the transmitted field. An index labels a spatial harmonic, not a separate incident source.


:::


(propagating-only-scattering-basis)=

### Can we solve using only propagating orders?

(can-we-solve-using-only-propagating-orders)=

**Keeping only the propagating part of a solved field is different from solving with only propagating orders.** The second choice can give inaccurate coefficients even for orders that propagate.

In a homogeneous exterior region, every retained outgoing propagating plane wave satisfies Helmholtz. Their sum therefore satisfies it too. The coefficients must also satisfy the **material-interface boundary conditions**. Near the patterned layer, evanescent harmonics help represent the required spatial variation. Omitting them during the scattering solve can change even the propagating coefficients that survive far from the grating.

Compare the two operations:

| Operation | What it means |
| --- | --- |
| Solve with a converged basis, then keep propagating output orders | Reconstruct the propagating part of the solved field. Close to the surface, this omits evanescent near-field structure. |
| Solve using only the exterior propagating-order labels | Solve a restricted approximation to the scattering problem; even its propagating coefficients may be inaccurate. |

Special cases, such as a uniform medium or an unpatterned planar interface illuminated by a propagating plane wave, need not excite additional evanescent orders. A patterned grating generally does. Inside the structured layer, the retained Fourier harmonics are coupled; “propagating” here classifies their **exterior** orders, not the layer eigenmodes.

### Small numerical example: same grating, different basis

Keep the physical example above fixed. In the air exterior, $|m|\lambda_0/d<1$ permits $m=0,\pm1,\pm2$: five propagating orders. Thus $M=2$ retains all propagating exterior labels but no evanescent exterior labels. Compare this solve with $M=24$ and $48$, which also retain evanescent harmonics.

The table below compares the **same complex transmitted coefficient** $C_2$ at the exit plane $z=-h$, with incident amplitude one and phase zero at $z=0$. It also compares total transmitted power. The $M=48$ result is a finer numerical reference, not an exact solution or a universal convergence guarantee. This optional experiment uses the same solver with different numerical cutoffs.

In [ ]:
from dataclasses import replace
from wave_optics.absorbing_grating import AbsorbingGratingConfiguration, solve_absorbing_grating

basis_example_config = AbsorbingGratingConfiguration(M=24)
basis_examples = {M: solve_absorbing_grating(replace(basis_example_config, M=M))
                  for M in (2, 24, 48)}
reference_C2 = basis_examples[48].transmission[48+2]
print(' M   N=2M+1        C_2 at exit             relative C_2 difference       T')
for M, trial in basis_examples.items():
    C2 = trial.transmission[M+2]
    difference = abs(C2-reference_C2)/abs(reference_C2)
    print(f'{M:2d}   {2*M+1:3d}     {C2.real:+.6f}{C2.imag:+.6f}j'
          f'                {difference:.3%}                 {trial.transmittance:.6f}')

:::{dropdown} What does the comparison show?

The propagating-only label set ($M=2$) changes $C_2$ by about **33.9%** relative to $M=48$, although total transmittance changes by only about **1.1%**. At $M=24$, the corresponding coefficient difference is about **0.02%**. A seemingly stable total power can hide a substantial error in an individual complex order.

The omitted exterior orders carry no individual outgoing normal power in lossless air, yet including their harmonics in boundary matching changes the propagating solution. This is why $M$ is an important **numerical parameter**, distinct from physical pitch, thickness or wavelength. Check convergence for the coefficients or near-field region you actually need; excellent power balance alone is insufficient.

:::

See [Appendix 1, numerical validation](Appendix_1_Fourier_Modal_Method_for_Scalar_Grating_Scattering.ipynb#fmm-validation-checks) for the distinction between boundary matching, physical-limit checks and truncation convergence.


## Checkpoint: from physical inputs to a field

**Material geometry + incident illumination + interface and outgoing conditions → solved complex field.**

1. Why is the material map insufficient to determine $U$?
2. Why can the field above the grating differ from the incident plane wave?
3. Why do the Fourier harmonics interact inside the layer but evolve independently in air?

:::{dropdown} Check your answers

1. We must specify the incoming wave and the conditions for matching fields and letting them leave the structure.
2. Above the grating, the incident and reflected fields add.
3. Inside the periodic material, the varying permittivity couples the harmonics. In uniform air, that coupling disappears.

The spatial wave equation and the coupled harmonic equations describe the same scattering problem.

:::

The field calculation retains 49 harmonics, while the order selector displays $-7\leq m\leq7$. Selecting a harmonic changes what we inspect, not the computed field.

:::{dropdown} Resetting and rerunning the views

Each explorer has its own reset. Rerunning a view replaces it. Rerunning setup closes the previous views before rebuilding the solution.
:::

You can continue once you can explain those observations in words. The matrix derivation is optional background for understanding the calculation, not an entrance requirement for the later plane-wave lessons.

**Next question:** how can the computed complex field be represented in terms of individual diffraction orders? Continue to [0.6: From complex fields to diffraction orders](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb).

[Previous: 0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb) · [Chapter index](0_0_Foundations_of_Scalar_Optics.ipynb)


## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete {download}`scripts download <downloads/grating_physics_scripts.zip>` to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. What physical inputs select the field?

**Script:** [0_5_01_material_geometry.py](Wave_solver_py_scripts/scripts/chapter_0/0_5_01_material_geometry.py)

Sample the absorbing grating geometry on a numerical grid.

### 2. What field does the solver calculate?

**Script:** [0_5_02_solve_and_power_balance.py](Wave_solver_py_scripts/scripts/chapter_0/0_5_02_solve_and_power_balance.py)

Solve a physical grating and inspect its power budget.

### 3. Can we solve using only propagating orders?

**Script:** [0_5_03_basis_convergence.py](Wave_solver_py_scripts/scripts/chapter_0/0_5_03_basis_convergence.py)

Compare a solve that retains only propagating-order labels with solves that also retain evanescent harmonics.

### 4. How does the same problem look in Fourier space?

**Script:** [0_5_04_explicit_harmonic_sum.py](Wave_solver_py_scripts/scripts/chapter_0/0_5_04_explicit_harmonic_sum.py)

Rebuild the solved field with an explicit Fourier sum.

**Numerical cutoffs:** the physical worked scripts start at $M=8$ (17 harmonics), whereas this teaching notebook uses $M=24$ (49 harmonics). The physical inputs agree, but numerical values can differ with truncation. To match the notebook, set `M = 24` in the script and retain adequate sampling; verify convergence for the quantity of interest. This does not change either default. See the [cutoff guide](Wave_solver_py_scripts/scripts/INDEX.md#chapter-0-cutoffs).




(refs-0-5)=

## References

(ref-0-5-maystre)=

[1] D. Maystre, “Analytic Properties of Diffraction Gratings,” in *Gratings: Theory and Numeric Applications*, 2nd revisited ed., E. Popov, Ed. Institut Fresnel, 2014, ch. 2. [Online]. Available: [Maystre, §§2.2–2.3](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=33).

**Reading notes:** Chapter 2 in E. Popov (ed.), *Gratings: Theory and Numeric Applications*, second revisited edition (2014). Section 2.2 develops the incident field, interface conditions and radiation conditions. Start with the geometry and the TE scalar reduction.

**Access:** Free institutional book chapter; not public domain. **Read for:** Read Sections 2.3–2.5: order expansion and energy balance.

(ref-0-5-lalanne-hugonin)=

[2] P. Lalanne and J.-P. Hugonin, “Fourier modal method for the analysis of optical nano-devices,” *Frontiers in Optics*, paper FWC1, 2006. [Online]. Available: [source](https://opg.optica.org/abstract.cfm?uri=FiO-2006-FWC1).

**Reading notes:** A short reference for the FMM/RCWA terminology used in Section 3.

## Further Reading

(ref-0-5-li-fmm)=

- L. Li, “Fourier Modal Method,” in *Gratings: Theory and Numeric Applications*, 2nd revisited ed., E. Popov, Ed., 2014, ch. 13. [Online]. Available: [PDF](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=541).

  **Reading notes:** Chapter 13 in the same volume. Sections 13.1–13.2 explain Fourier expansion, layer modes and interface matching. This is advanced electromagnetic background; our notebook implements the stated scalar model.

(ref-0-5-s4)=

- V. Liu and S. Fan, “S4: A free electromagnetic solver for layered periodic structures,” *Computer Physics Communications*, vol. 183, pp. 2233–2244, 2012. [Online]. Available: [PDF](https://web.stanford.edu/group/fan/publication/Liu_ComputerPhysicsCommunications_183_2233_2012.pdf). doi: 10.1016/j.cpc.2012.04.026.

  **Reading notes:** **Access:** Free author-institution PDF; copyright retained. **Read for:** Optional advanced comparison: vector FMM, not our scalar solver.

## Reference notes

These readings support the physical setup and optional solver derivation; they are not prerequisites.

Freely accessible works are not necessarily in the public domain.

**Convention check:** Translate each source's order signs, propagation side, polarization assumptions and amplitude/power normalization to this notebook's stated model.


